Absolutely. Let's implement Stage 1.4.2.5.4 hands-on in your existing VS Code notebook and keep it focused on Docling, not manual OCR.

### Stage 1.4.2.5.4 — Docling Table / Image / Formula Understanding
The objective is to answer:

When Docling receives a complex PDF, how does it recognize and represent tables, images/figures, and formulas?
Our flow will be:

                     PDF
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
        Table       Image       Formula
          │           │           │
          ↓           ↓           ↓
       Docling     Docling      Docling
          │           │           │
          └───────────┼───────────┘
                      ↓
               DoclingDocument
                      ↓
        Structured representation
I've also created a dedicated test PDF for this stage.
Download the Stage 1.4.2.5.4 sample PDF
It contains:

a structured table
an architecture diagram as an embedded image
a mathematical formula
captions and explanatory text
### Step 1 — Verify your Docling version
You already told me you're using Docling 2.120.3, but let's verify directly in the notebook.

In [133]:
import docling

print("Docling version:", docling.__version__)

Docling version: 2.120.3


Expected:

Docling version: 2.120.3

### Step 2 — Define the sample PDF
Use the PDF we just generated.

In [134]:
from pathlib import Path

pdf_path = Path(
    "D:\\AI Learning\\rag-learning\\data\\raw\\pdf\\Stage 1.4.2.5.4-docling_table_image_formula_sample.pdf"
)

print(pdf_path)
print(pdf_path.exists())

D:\AI Learning\rag-learning\data\raw\pdf\Stage 1.4.2.5.4-docling_table_image_formula_sample.pdf
True


You should get:

stage_1_4_2_5_4/docling_table_image_formula_sample.pdf
    
    True

If you place the PDF somewhere else in your RAG project's data/raw/pdf/ directory, simply change the path.

### Step 3 — Import Docling

In [135]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

### Step 4 — Convert the PDF

In [136]:
result = converter.convert(pdf_path)

doc = result.document

[INFO] 2026-08-20 17:11:27,813 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-20 17:11:27,815 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-20 17:11:27,844 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-20 17:11:27,845 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-20 17:11:28,053 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-20 17:11:28,055 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-20 17:11:28,059 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-20 17:11:28,061 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobil

Now we have:

    PDF
    ↓
    DocumentConverter
    ↓
    ConversionResult
    ↓
    DoclingDocument

Remember this distinction:

result

is the conversion result.

    result.document

is the actual:

    DoclingDocument

### Step 5 — First inspect the Markdown
We deliberately do this, but we will not rely on it as our only inspection method.

In [137]:
markdown_text = doc.export_to_markdown()

print(markdown_text)

## Stage 1.4.2.5.4 - Docling Table / Image / Formula Understanding

This document is intentionally designed as a controlled test document for Docling. It contains ordinary text, a structured table, an embedded architecture image, and a mathematical formula.

## 1. Structured Table

|   Stage | Component           | Purpose                              |
|---------|---------------------|--------------------------------------|
|       1 | Event Producer      | Publishes business events            |
|       2 | Azure Event Hubs    | Ingests and partitions event streams |
|       3 | Stream Consumer     | Reads and transforms events          |
|       4 | Azure Data Explorer | Stores data for analytics            |

## 2. Architecture Image / Figure

## Event Processing Architecture

Figure 1 - Event processing architecture

<!-- image -->

## 3. Mathematical Formula

The following mathematical expression is included as an image so that we can separately observe how Docling treats formula-

You should see the ordinary text and table represented in Markdown.

For example, the table should look approximately like:

| Stage | Component | Purpose |
|---|---|---|
| 1 | Event Producer | Publishes business events |
| 2 | Azure Event Hubs | Ingests and partitions event streams |
...

The picture may appear as a placeholder depending on the export settings.

Docling's export_to_markdown() supports image modes such as placeholder, embedded, and referenced images. (Docling Project)

### Step 6 — Inspect the tables directly ⭐
This is where our learning becomes more interesting.

Run:

In [138]:
print("Number of tables:", len(doc.tables))

Number of tables: 1


Then:

In [139]:
for i, table in enumerate(doc.tables):
    print(f"\n===== TABLE {i + 1} =====")
    print(table.export_to_markdown(doc))


===== TABLE 1 =====
|   Stage | Component           | Purpose                              |
|---------|---------------------|--------------------------------------|
|       1 | Event Producer      | Publishes business events            |
|       2 | Azure Event Hubs    | Ingests and partitions event streams |
|       3 | Stream Consumer     | Reads and transforms events          |
|       4 | Azure Data Explorer | Stores data for analytics            |


We are no longer asking:

    "What did Markdown export give me?"
We're asking:

    "What table objects did Docling actually create?"
Docling exposes table items and supports exporting an individual table to Markdown, HTML, and other structured representations.

### Step 7 — Export the table as a DataFrame
This is extremely useful for RAG later.

In [140]:
for i, table in enumerate(doc.tables):
    print(f"\n===== TABLE {i + 1} =====")

    df = table.export_to_dataframe(doc)

    display(df)


===== TABLE 1 =====


,Stage,Component,Purpose
0,1,Event Producer,Publishes business events
1,2,Azure Event Hubs,Ingests and partitions event streams
2,3,Stream Consumer,Reads and transforms events
3,4,Azure Data Explorer,Stores data for analytics


Conceptually:

    PDF Table
    ↓
    Docling TableItem
    ↓
    DataFrame
    
You should get something similar to:
| Stage | Component           | Purpose                              |
| ----: | ------------------- | ------------------------------------ |
|     1 | Event Producer      | Publishes business events            |
|     2 | Azure Event Hubs    | Ingests and partitions event streams |
|     3 | Stream Consumer     | Reads and transforms events          |
|     4 | Azure Data Explorer | Stores data for analytics            |

This is much more powerful than simply extracting text.

### Step 8 — Understand the table structure
Now inspect the first table:

In [141]:
table = doc.tables[0]

print(type(table))

<class 'docling_core.types.doc.items.table.table.TableItem'>


Then:

In [142]:
print(table)

self_ref='#/tables/0' parent=RefItem(cref='#/body') children=[] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.TABLE: 'table'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=62.32006411062664, t=652.1923482646223, r=532.2838882782004, b=540.2097568423433, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))] source=[] comments=[] captions=[] references=[] footnotes=[] image=None data=TableData(table_cells=[TableCell(bbox=BoundingBox(l=68.77953, t=198.76710000000014, r=91.92502999999999, b=206.6296000000001, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=0, end_col_offset_idx=1, text='Stage', column_header=True, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=119.80315, t=198.76710000000014, r=167.02065, b=206.6296000000001, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx

And inspect its available methods:

In [143]:
[m for m in dir(table) if not m.startswith("_")]

['add_annotation',
 'annotations',
 'caption_text',
 'captions',
 'children',
 'comments',
 'construct',
 'content_layer',
 'copy',
 'data',
 'dict',
 'export_to_dataframe',
 'export_to_doctags',
 'export_to_document_tokens',
 'export_to_html',
 'export_to_markdown',
 'export_to_otsl',
 'footnotes',
 'from_orm',
 'get_annotations',
 'get_image',
 'get_location_tokens',
 'get_ref',
 'image',
 'json',
 'label',
 'meta',
 'model_computed_fields',
 'model_config',
 'model_construct',
 'model_copy',
 'model_dump',
 'model_dump_json',
 'model_extra',
 'model_fields',
 'model_fields_set',
 'model_json_schema',
 'model_parametrized_name',
 'model_post_init',
 'model_rebuild',
 'model_validate',
 'model_validate_json',
 'model_validate_strings',
 'parent',
 'parse_file',
 'parse_obj',
 'parse_raw',
 'prov',
 'references',
 'schema',
 'schema_json',
 'self_ref',
 'source',
 'update_forward_refs',
 'validate']

Pay particular attention to methods related to:

    export
    cells
    captions
    image
    provenance

The important concept is:

    TableItem
    │
    ├── rows
    ├── columns
    ├── cells
    ├── caption
    ├── provenance
    └── structure

This is why table-aware ingestion is different from ordinary text extraction.

### Step 9 — Now investigate images / pictures
Run:

In [144]:
print("Number of pictures:", len(doc.pictures))

Number of pictures: 1


Then:

In [145]:
for i, picture in enumerate(doc.pictures):
    print(f"\n===== PICTURE {i + 1} =====")
    print(type(picture))
    print(picture)


===== PICTURE 1 =====
<class 'docling_core.types.doc.items.picture.picture.PictureItem'>
self_ref='#/pictures/0' parent=RefItem(cref='#/body') children=[RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9')] content_layer=<ContentLayer.BODY: 'body'> meta=None label=<DocItemLabel.PICTURE: 'picture'> prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=89.04425054473585, t=421.15332353903943, r=452.5822882712545, b=302.1633793329843, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))] source=[] comments=[] captions=[RefItem(cref='#/texts/5')] references=[] footnotes=[] image=None annotations=[]


This tells us whether Docling detected our architecture diagram as a PictureItem.
Docling provides PictureItem support and can retrieve the corresponding image from the DoclingDocument. (Docling Project)

### Step 10 — Extract the detected image
If we have at least one picture:

In [146]:
if len(doc.pictures) > 0:

    picture = doc.pictures[0]

    picture_image = picture.get_image(doc)

    print(type(picture_image))

<class 'NoneType'>


The above code produces:

<class 'NoneType'>

This does not mean that Docling failed to detect your architecture diagram.

It means:

Docling knows that a picture exists, but we didn't ask the PDF pipeline to generate the actual picture image.

This distinction is extremely important.

If the image is available, save it:

In [147]:
from pathlib import Path

if picture_image is not None:
    # Define the target output path
    output_file = Path("output/Stage 1.4.2.5.4/docling_extracted_picture.png")

    # Create parent directories if they do not exist
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Save the extracted picture
    picture_image.save(output_file)
    print("Picture saved.")

Now you've demonstrated:

PDF
 ↓
Embedded architecture diagram
 ↓
Docling
 ↓
PictureItem
 ↓
PIL Image
 ↓
PNG
This is very different from OCR.

### Step 11 — Important distinction: Image Detection vs Image 

Understanding

This is one of the most important concepts in this stage.
Suppose Docling tells us:

        PictureItem
That means:

        Docling identified a picture/figure region.
It does not necessarily mean:

        "Docling understands that this is an Azure Event Hubs → Stream Consumer → Data Explorer architecture."
Those are two different levels:

    Level 1
    Image detection
            ↓
    "There is a picture here."
        
    versus:

    Level 2
    Image understanding
            ↓
    "This is an event-processing architecture."

For the second task, vision-language capabilities can be involved. Docling supports picture classification and picture-description options in its pipeline configuration.

That distinction will become extremely important when we reach your multimodal RAG stage.

### Step 12 — Investigate the formula
Our sample PDF contains:

P(A|B) = P(B|A)P(A) / P(B)

The formula is deliberately included as an image for this first experiment.

Let's inspect the document's text items.

In [148]:
for item, level in doc.iterate_items():

    print(
        type(item).__name__,
        getattr(item, "label", None),
        getattr(item, "text", "")
    )

SectionHeaderItem section_header Stage 1.4.2.5.4 - Docling Table / Image / Formula Understanding
TextItem text This document is intentionally designed as a controlled test document for Docling. It contains ordinary text, a structured table, an embedded architecture image, and a mathematical formula.
SectionHeaderItem section_header 1. Structured Table
TableItem table 
SectionHeaderItem section_header 2. Architecture Image / Figure
SectionHeaderItem section_header Event Processing Architecture
PictureItem picture 
TextItem caption Figure 1 - Event processing architecture
SectionHeaderItem section_header 3. Mathematical Formula
TextItem text The following mathematical expression is included as an image so that we can separately observe how Docling treats formula-like visual content.
FormulaItem formula 
TextItem caption Figure 2 - Conditional probability formula
TextItem text Experiment: We will inspect Docling's detected tables, pictures, and formula/text elements independently instead 

Look for anything whose label indicates:

FORMULA
The exact result is something we want to observe, not assume.

### Step 13 — Understand formula enrichment
This is where Docling becomes more interesting.

Docling's PDF pipeline has a do_formula_enrichment option for mathematical formula recognition and conversion to LaTeX.

Conceptually:

PDF
    ↓
    Formula region
    ↓
    Formula recognition
    ↓
    LaTeX
For example, ideally:

P(A|B) = P(B|A)P(A) / P(B)

could become something similar to:

P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}

However, we should not turn this option on blindly in your Docling 2.120.3 environment.

Your installed version is important, and formula enrichment can require additional model dependencies/resources. The current Docling documentation confirms that formula enrichment is a specialized processing option and that enabling multiple enrichment features increases processing time.

So our first experiment is intentionally:

    Detect and inspect first → enable specialized enrichment only after we understand the baseline.

### Step 14 — Inspect all document elements together
Now let's create one useful diagnostic cell:

In [149]:
for item, level in doc.iterate_items():

    label = getattr(item, "label", None)

    text = getattr(item, "text", "")

    print(
        f"Type={type(item).__name__:<20} "
        f"Label={str(label):<20} "
        f"Text={text[:100]!r}"
    )

Type=SectionHeaderItem    Label=section_header       Text='Stage 1.4.2.5.4 - Docling Table / Image / Formula Understanding'
Type=TextItem             Label=text                 Text='This document is intentionally designed as a controlled test document for Docling. It contains ordin'
Type=SectionHeaderItem    Label=section_header       Text='1. Structured Table'
Type=TableItem            Label=table                Text=''
Type=SectionHeaderItem    Label=section_header       Text='2. Architecture Image / Figure'
Type=SectionHeaderItem    Label=section_header       Text='Event Processing Architecture'
Type=PictureItem          Label=picture              Text=''
Type=TextItem             Label=caption              Text='Figure 1 - Event processing architecture'
Type=SectionHeaderItem    Label=section_header       Text='3. Mathematical Formula'
Type=TextItem             Label=text                 Text='The following mathematical expression is included as an image so that we can separately 

This is one of the most important cells in this stage.

We are beginning to see:

    DoclingDocument
    │
    ├── TextItem
    ├── SectionHeaderItem
    ├── TableItem
    ├── PictureItem
    ├── ...
    └── Formula-related element, if detected

Now we're actually learning the Docling document model, rather than treating Docling as a black-box PDF-to-Markdown converter.

### Step 15 — Export the complete structured document
Since you previously encountered:

    AttributeError:
    DoclingDocument has no attribute export_to_json

we must remember the correct API for your Docling v2 workflow:

document_dict = doc.export_to_dict()

Docling v2 moved document export operations onto DoclingDocument; the official v2 documentation explicitly shows export_to_dict(), export_to_markdown(), and export_to_document_tokens(). 

You can inspect:
    document_dict.keys()
    
and save it:

In [150]:
document_dict = doc.export_to_dict()
document_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [151]:
import json

output_json = Path(
    "output/docling_table_image_formula.json"
)

output_json.write_text(
    json.dumps(
        document_dict,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(output_json)

output\docling_table_image_formula.json


This JSON becomes very useful for debugging.

### Step 16 — Our first Table / Image / Formula summary

After running the notebook, create this summary:

In [152]:
print("========== DOCLING CONTENT SUMMARY ==========")

print("Tables   :", len(doc.tables))
print("Pictures :", len(doc.pictures))

formula_items = []

for item, level in doc.iterate_items():

    label = str(getattr(item, "label", ""))

    if "formula" in label.lower():
        formula_items.append(item)

print("Formulas :", len(formula_items))

========== DOCLING CONTENT SUMMARY ==========
Tables   : 1
Pictures : 1
Formulas : 1


Ideally we'll get something like:

========== DOCLING CONTENT SUMMARY ==========

Tables   : 1

Pictures : 2

Formulas : ...

Don't worry if the formula count isn't what we expect yet. That itself is part of the experiment: our formula is intentionally an image, so the baseline pipeline may treat it as a picture rather than as a semantic formula.

### Step 17 — What we've learned from this experiment

The most important mental model is:

                    PDF
                     │
                     ▼
                  Docling
                     │
          ┌──────────┼──────────┐
          ↓          ↓          ↓
       Table      Picture     Formula
          │          │          │
          ↓          ↓          ↓
      TableItem  PictureItem  Formula/
                              Text item
          │          │          │
          └──────────┼──────────┘
                     ↓
              DoclingDocument
And each content type needs a different downstream strategy.

| Content | What we want from Docling    | RAG implication       |
| ------- | ---------------------------- | --------------------- |
| Text    | Text + hierarchy             | Normal text chunk     |
| Table   | Rows/columns/cells           | Structure-aware chunk |
| Image   | Picture + location           | Vision processing     |
| Formula | Formula/LaTeX representation | Formula-aware text    |
| Caption | Association with element     | Preserve with element |

### One important correction from our previous stage

This stage also helps explain the Analytics Layer problem you found.

We should never assume:

        Looks like a table
                ↓
        Must be TableItem

or:

Looks like two columns
        ↓
Must be two independent text regions

The PDF's internal structure matters.

That's exactly why we're now explicitly inspecting:

        doc.tables
        doc.pictures
        doc.iterate_items()
        doc.export_to_dict()

rather than only:

doc.export_to_markdown()

### Where we are after this implementation

Our Docling track now becomes:

                Stage 1.4.2.5
                OCR Approach
                │
                ├── 1.4.2.5.1 — Basic OCR
                │   ✅
                │
                ├── 1.4.2.5.2 — Docling for Scanned PDFs
                │   ✅
                │
                ├── 1.4.2.5.3 — Docling Layout & Reading Order
                │   ✅
                │
                ├── 1.4.2.5.4 — Table / Image / Formula Understanding
                │   🔵 CURRENT
                │
                └── 1.4.2.5.5 — Docling → LangChain → RAG
    ⏳
Do not move to 1.4.2.5.5 yet.
First, run the notebook cells above against the new PDF. The important outputs for our next step will be:

1. len(doc.tables)
2. len(doc.pictures)
3. table.export_to_markdown(doc)
4. table.export_to_dataframe(doc)
5. picture.get_image(doc)
6. output of doc.iterate_items()
7. formula-related items, if any
8. doc.export_to_markdown()

That will let us investigate exactly how your Docling 2.120.3 installation represents each content type, rather than relying on assumptions from newer Docling documentation.